# 01 - Module 与 Parameter 基础

## 学习目标

1. 理解 `nn.Module` 是什么，以及它内部的 8 个有序字典
2. 掌握创建模型的基本流程：`__init__` + `forward`
3. 理解 `nn.Parameter` 与普通 Tensor 的区别
4. 学会使用 `named_parameters()` 遍历模型参数

---

In [ ]:
import torch
import torch.nn as nn

## 1. 什么是 nn.Module？

`nn.Module` 是 PyTorch 中所有神经网络模块的**基类**。你写的每个网络层、每个模型都应该继承它。

它内部维护了 **8 个有序字典（OrderedDict）**，用来管理模型的各种组件：

| 属性 | 存储内容 |
|------|----------|
| `_parameters` | 模型自身的参数（`nn.Parameter`） |
| `_buffers` | 缓冲区（如 BN 的 running_mean） |
| `_modules` | 子模块（子层） |
| `_backward_hooks` | 反向传播钩子 |
| `_forward_hooks` | 前向传播钩子 |
| `_forward_pre_hooks` | 前向传播前钩子 |
| `_state_dict_hooks` | state_dict 钩子 |
| `_load_state_dict_pre_hooks` | 加载 state_dict 前钩子 |

让我们通过代码来验证这些属性的存在。

In [ ]:
# 查看 nn.Module 内部的 8 个有序字典
module = nn.Module()

# 打印所有以 _ 开头的 dict 类型属性
for attr_name in dir(module):
    attr = getattr(module, attr_name)
    if isinstance(attr, dict) and attr_name.startswith('_'):
        print(f"{attr_name}: {type(attr).__name__}")

**关键理解：**

- `_parameters`、`_buffers`、`_modules` 是最常用的三个
- 当你在 `__init__` 中设置 `self.conv = nn.Conv2d(...)` 时，PyTorch 会自动将该子模块注册到 `_modules` 中
- 当你设置 `self.weight = nn.Parameter(...)` 时，会自动注册到 `_parameters` 中

## 2. 创建一个简单模型：TinnyCNN

创建模型只需两步：
1. **`__init__`**：定义网络层（构建子模块）
2. **`forward`**：定义前向传播逻辑（数据如何流经网络）

In [ ]:
class TinnyCNN(nn.Module):
    """一个简单的卷积神经网络示例。"""

    def __init__(self):
        super().__init__()
        # __init__ 中定义各网络层
        self.conv1 = nn.Conv2d(1, 6, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(6 * 14 * 14, 10)

    def forward(self, x):
        """前向传播：定义数据如何流经网络。"""
        x = self.pool(torch.relu(self.conv1(x)))  # 卷积 -> 激活 -> 池化
        x = x.view(x.size(0), -1)                 # 展平
        x = self.fc1(x)                            # 全连接
        return x


# 实例化模型
model = TinnyCNN()
print(model)

In [ ]:
# 测试前向传播
# 输入：batch_size=2, channels=1, height=28, width=28
x = torch.randn(2, 1, 28, 28)
output = model(x)
print(f"输入形状: {x.shape}")
print(f"输出形状: {output.shape}")

**关键观察：**

1. 必须调用 `super().__init__()` 来初始化基类
2. 使用 `model(x)` 而不是 `model.forward(x)` —— 前者会触发钩子等机制
3. `print(model)` 会打印模型结构，这是 `nn.Module` 的 `__repr__` 方法

## 3. 查看 `_modules` 字典

当我们在 `__init__` 中设置子模块时，PyTorch 自动将它们注册到 `_modules` 中。

In [ ]:
# 查看 _modules 字典
print("_modules 中注册的子模块:")
for name, module in model._modules.items():
    print(f"  {name}: {module}")

print(f"\n子模块数量: {len(model._modules)}")

## 4. nn.Parameter vs 普通 Tensor

`nn.Parameter` 是 `Tensor` 的子类，区别在于：

| 特性 | `nn.Parameter` | 普通 `Tensor` |
|------|---------------|---------------|
| `requires_grad` | 默认 `True` | 默认 `False` |
| 自动注册到 `_parameters` | 是 | 否 |
| `parameters()` 可遍历 | 是 | 否 |
| 会被 `state_dict()` 保存 | 是 | 否 |

In [ ]:
# 对比 nn.Parameter 和普通 Tensor
class DemoModule(nn.Module):
    def __init__(self):
        super().__init__()
        # nn.Parameter: 会被自动注册
        self.param_weight = nn.Parameter(torch.randn(3, 3))
        # 普通 Tensor: 不会被注册
        self.normal_tensor = torch.randn(3, 3)

    def forward(self, x):
        return x


demo = DemoModule()

print("=== nn.Parameter ===")
print(f"类型: {type(demo.param_weight)}")
print(f"requires_grad: {demo.param_weight.requires_grad}")

print("\n=== 普通 Tensor ===")
print(f"类型: {type(demo.normal_tensor)}")
print(f"requires_grad: {demo.normal_tensor.requires_grad}")

In [ ]:
# 查看 _parameters 字典：只有 nn.Parameter 会被注册
print("_parameters 中注册的参数:")
for name, param in demo._parameters.items():
    print(f"  {name}: shape={param.shape}")

print(f"\nnormal_tensor 在 _parameters 中吗？{'normal_tensor' in demo._parameters}")

In [ ]:
# state_dict 只包含 Parameter，不包含普通 Tensor
print("state_dict 的 keys:")
for key in demo.state_dict():
    print(f"  {key}")

print("\n注意: normal_tensor 不在 state_dict 中！")

**重要结论：**

如果你希望一个张量作为模型的**可学习参数**，必须用 `nn.Parameter` 包装它。否则它不会被优化器更新，也不会被 `state_dict()` 保存。

## 5. 遍历模型参数

PyTorch 提供了多种方式来遍历模型参数。

In [ ]:
model = TinnyCNN()

# 方式 1: parameters() - 只返回参数张量
print("=== parameters() ===")
for i, param in enumerate(model.parameters()):
    print(f"参数 {i}: shape={param.shape}, requires_grad={param.requires_grad}")

In [ ]:
# 方式 2: named_parameters() - 返回 (名称, 参数) 对
print("=== named_parameters() ===")
for name, param in model.named_parameters():
    print(f"{name}: shape={param.shape}")

In [ ]:
# 统计模型的总参数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")

**关键观察：**

1. `parameters()` 会递归地遍历所有子模块的参数
2. `named_parameters()` 的名称格式为 `子模块名.参数名`，如 `conv1.weight`
3. 统计参数量时用 `p.numel()` 获取参数中元素的总数

## 6. `__setattr__` 的自动注册机制

`nn.Module` 重写了 `__setattr__`，所以当你在 `__init__` 中做 `self.xxx = ...` 时：
- 如果值是 `nn.Parameter` → 注册到 `_parameters`
- 如果值是 `nn.Module` → 注册到 `_modules`
- 其他类型 → 普通属性

In [ ]:
# 验证自动注册机制
class AutoRegisterDemo(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(3, 2)              # -> _modules
        self.my_param = nn.Parameter(torch.ones(5)) # -> _parameters
        self.my_list = [1, 2, 3]                    # -> 普通属性

    def forward(self, x):
        return x


demo = AutoRegisterDemo()

print("_modules:", list(demo._modules.keys()))
print("_parameters:", list(demo._parameters.keys()))
print("my_list:", demo.my_list, "（普通属性，不在字典中）")

## 7. 练习题

### 练习 1：创建一个简单的全连接网络

创建一个包含两个全连接层的网络：
- 输入维度: 784（28x28 展平）
- 隐藏层: 256，使用 ReLU 激活
- 输出层: 10

In [ ]:
class SimpleFC(nn.Module):
    """简单的全连接网络。"""

    def __init__(self):
        super().__init__()
        # TODO: 定义两个全连接层
        pass

    def forward(self, x):
        # TODO: 实现前向传播
        pass


# 测试代码
# model = SimpleFC()
# x = torch.randn(2, 784)
# output = model(x)
# print(f"输出形状: {output.shape}")  # 应为 [2, 10]
# print(f"参数量: {sum(p.numel() for p in model.parameters()):,}")

### 练习 2：自定义 Parameter

创建一个模块，包含一个可学习的缩放因子 `scale`（使用 `nn.Parameter`），将输入乘以该因子。

In [ ]:
class ScaleLayer(nn.Module):
    """可学习的缩放层。"""

    def __init__(self, dim):
        super().__init__()
        # TODO: 定义一个 nn.Parameter 作为缩放因子
        pass

    def forward(self, x):
        # TODO: 将输入乘以缩放因子
        pass


# 测试代码
# layer = ScaleLayer(dim=5)
# print(f"scale 参数: {layer.scale}")
# print(f"是否在 parameters() 中: {'scale' in dict(layer.named_parameters())}")
# x = torch.ones(2, 5)
# print(f"输出: {layer(x)}")

## 8. 小结

### 核心概念

- **`nn.Module`** 是所有模型的基类，内部用 8 个有序字典管理参数、子模块、钩子等
- **创建模型两步**：`__init__` 定义层，`forward` 定义数据流
- **`nn.Parameter`** 是特殊的 Tensor，默认 `requires_grad=True`，会被自动注册

### 关键方法

| 方法 | 作用 |
|------|------|
| `parameters()` | 遍历所有参数 |
| `named_parameters()` | 遍历所有参数（带名称） |
| `state_dict()` | 获取模型参数字典 |

### 易错点

1. 忘记调用 `super().__init__()`
2. 使用普通 Tensor 而非 `nn.Parameter` 作为可学习参数
3. 直接调用 `model.forward(x)` 而非 `model(x)`

### 下一步

在下一个 notebook 中，我们将学习 **容器模块**（Sequential、ModuleList、ModuleDict），用来更灵活地组织网络结构。